# B2-t5small — `T.Oc138k.compact3.e6 → Oc`: the control for B2

This is the same run as `25_B2_conditions_compoundt5_3fields.ipynb` with one thing changed:
the base is `t5-small` instead of `sagawa/CompoundT5`. Same 138,869 training rows, same three
fields, same compact serialization, same learning rate, same effective batch, same evaluation
on the same leak-free 5,687-record test set.

**Why it has to be re-run rather than quoted.** RESULTS.md already carries t5-small numbers
(solvent 29.5 / 49.5, catalyst 22.3 / 31.5 strict top-1 / top-5, and so on), but those came
from **JSON targets over four fields**. Comparing them against a compact three-field
CompoundT5 run would mix three differences — base, serialization, field set — into one number
and attribute all of it to the base. After this run the base is the only difference left.

The serialization is not neutral for either side. Measured over 2000 real rows, mean target
tokens and `<unk>` per row: t5-small JSON 60.9 / 2.0, t5-small compact 20.7 / 0. So t5-small
also gains from the compact form, and the honest control has to give it that gain too.

**Six epochs.** t5-small is 60M against CompoundT5's 220M and trains roughly four times
faster per epoch, so the run that costs CompoundT5 four epochs buys t5-small six. Both stop
at their own `eval_loss` minimum via `load_best_model_at_end` rather than at a fixed epoch
count — the point is to compare converged models, not equal step counts.

**Per-record output is kept** for the same reason as B2: a paired McNemar between the two
bases needs both runs record by record, which is exactly what the earlier t5-small runs did
not save.

**No `yield_percent`.** Of 5,687 test records, B1's rank-1 generation emitted a yield number
61 times and abstained with `?` 5,234 times, though 60.7% of training rows carry one. The
field is not a function of `(product, reactants)` and both bases now drop it.

**Data:** `kuzmenkoiryna/retro-planner-ord-conditions`.

**Cost:** ~3 h 20 min training + ~15 min evaluation.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/v2_ord_conditions_test_clean.jsonl", recursive=True))
for path in (train_file, val_file, test_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "t5-small"
learning_rate = 5e-4   # same as B2 and as every earlier Model 2 run
condition_fields = "solvent,catalyst,temperature_celsius"
output_dir = "/kaggle/working/model2_conditions_t5small_3f"
time_budget_minutes = 200

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

# Batch 32 x 1, the script's default: t5-small is 60M and fits, so it does not need B2's
# 8 x 4 split. The effective batch is 32 either way, which is what keeps the two runs
# comparable -- the split exists only to fit a 220M model on a T4.
#
# --max-target-length 200 matches B2 rather than the old 64. Under t5-small's vocabulary the
# compact target is shorter than under CompoundT5's, so this truncates nothing; padding is
# dynamic, so the unused headroom costs nothing.
!torchrun --nproc_per_node=2 scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_t5small_work \
    --target-format compact \
    --condition-fields "{condition_fields}" \
    --max-target-length 200 \
    --learning-rate {learning_rate} \
    --num-train-epochs 6 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# t5-small's vocabulary already spells `|`, `?` and digits, so unlike B2 the repair should be
# close to a no-op here -- the compact target measured 0 <unk> per row under this tokenizer.
# Printed anyway: a surprise here would mean the two runs saw different targets.
!grep -E "new character token|Train examples|condition field" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"
marker = json.load(open(f"{output_dir}/final/conditions_format.json"))
print("format marker:", marker)
assert marker["fields"] == condition_fields.split(","), "marker disagrees with the requested fields"

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 12)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))
best_epoch = min(points, key=lambda p: p[1])[0]
print(f"  best at epoch {best_epoch:.2f} of {points[-1][0]:.2f} reached")

In [ ]:
# Batch 32 here against B2's 8, for the same memory reason as training. Beam count, generation
# length and the test file are identical, which is what the comparison rests on.
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --num-beams 10 --batch-size 32 --device cuda \
    --max-target-length 200 \
    --output "/kaggle/working/B2_conditions_t5small_3f_clean_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/B2_conditions_t5small_3f_clean_topk.json"))
print(json.dumps(data["summary"], indent=2))
print("per-record entries kept for a paired test:", len(data["records"]))